## Data-Level Security Using Dynamic Views 

Thi notebook demonstrates how data-level security  can be implemented using dynamic views

The implementation covers 2 key capabilities

- Row-level Security: Controls which rows are visible to a user
- Column-level Security: Controls how sensitive values are displayed

A simple sales dataset is used to illustrate how different users receive different results when querying the same views.

Step 1: Set up schema for the table

In [0]:
USE CATALOG demo;
CREATE SCHEMA IF NOT EXISTS data_security;

In [0]:
USE SCHEMA data_security;

**Step 2: Create the table**

A table is defined to represent sales data across multiple regions

In [0]:
CREATE OR REPLACE TABLE sales_dynamic(
    id INT,
    region STRING,
    email STRING,
    revenue INT
);

**Step 3: Insert the data**

The dataset includes records from both UK and US regions.

This enables validation of row-level security by ensuring that different users see only relevant regional data.

In [0]:
Step 3: Insert the data

In [0]:
INSERT INTO sales_dynamic VALUES
(1, 'UK', 'john.doe@email.com', 10000),
(2, 'US', 'mike.smith@email.com', 20000),
(3, 'UK', 'sara.jones@email.com', 15000),
(4, 'US', 'david.brown@email.com', 25000),
(5, 'UK', 'emma.wilson@email.com', 18000);

In [0]:
SELECT * FROM sales_dynamic;

**Step 4: Create the dynamic view**

The dynamic view implements both row-level security and column-level masking

**Row-Level Security**

- UK group -> returns only rows where region = 'UK'
- US group -> returns only rows where region = 'US'
- Admin group -> returns all rows

**Column-level Masking**

Admin group -> full email visible
Other users -> email is masked

**Key Function:is_account_group_member()**

This function evaluates whether the current user belongs to a specified group.

Examples

- is_account_group_member('uk-sg')
- is_account_group_member('us-sg')
- is_account_group_member('admin-sg')

The function returns TRUE or FALSE and is used to drive conditional logic inside the view.

In [0]:
SELECT is_account_group_member('admin-sg'),is_account_group_member('uk-sg');

In [0]:
CREATE OR REPLACE VIEW vw_sales_dynamic AS
SELECT id,
       region,
       CASE
        WHEN is_account_group_member('admin-sg') THEN email
        ELSE concat(substr(email, 1, 1), '***@', split(email,'@')[1])
       END AS email,
       revenue
FROM sales_dynamic
WHERE is_account_group_member('admin-sg')
OR (is_account_group_member('uk-sg') AND region = 'UK')
OR (is_account_group_member('us-sg') AND region = 'US');

Step 5: Query the dynamic view

In [0]:
SELECT * FROM vw_sales_dynamic;